# LLM Fine-Tuning Deep Dive, Part 3 of 3: Evaluate First, Compare Second, Decide Last

> **The Riverside decision:** which artifact, if any, has earned further testing for a specific Riverside job?

Parts 1 and 2 created artifacts. Part 3 does not begin by ranking them. Each candidate practiced a different job, so each candidate must first answer its own objective-aligned question on independent evidence.

## Evidence Boundary

Parts 1 and 2 used all 40 Aria chapters for mechanism-sized training. Therefore:

- those chapters and all examples derived from them are training-contaminated;
- same-corpus generations, phrase scores, and perplexity are diagnostics, not generalization evidence;
- a release claim requires a separately versioned suite that was not used for training, prompt design, rubric examples, or threshold tuning; and
- missing or contaminated required evidence blocks a decision.

This notebook can demonstrate every evaluation calculation with the available corpus, but it must label those calculations honestly.

## Vocabulary Used Throughout

| Term | Meaning in this notebook |
| --- | --- |
| **Baseline** | The accepted comparison point for one claim: usually the untouched model, or accepted SFT when evaluating DPO |
| **Candidate** | One immutable trained artifact being evaluated for one declared job |
| **Case** | One prompt, source passage, expected constraints, and any reviewer instructions |
| **Scorer** | The rule that turns one candidate response into one record, such as NLL, contract pass/fail, or blinded preference |
| **Metric** | An aggregation over case records, such as perplexity, complete-pass rate, or credited win rate |
| **Gate** | A product-owned threshold fixed before inspecting candidate results |
| **Supported** | All required evidence is valid and every required gate passes |
| **Not supported** | Valid required evidence exists, but at least one required gate fails |
| **Evidence unavailable** | Required evidence is missing, contaminated, or too uncertain to apply the gates |

`Evidence unavailable` names the actual problem: Riverside cannot yet evaluate the claim on valid required evidence.

## The Evaluation Technique: One Candidate at a Time

Every candidate follows the same vertical flow:

```mermaid
flowchart TD
    Job["1. State the candidate's job"] --> Claim["2. Write one falsifiable claim"]
    Claim --> Suite["3. Freeze an independent versioned case suite"]
    Suite --> Scorer["4. Choose a scorer that matches the claim"]
    Scorer --> Run["5. Run baseline and candidate on the same cases"]
    Run --> Records["6. Preserve case-level records and failure reasons"]
    Records --> Metric["7. Aggregate one declared metric"]
    Metric --> Gate["8. Apply precommitted gates"]
    Gate --> Conclusion{"9. Conclusion"}
    Conclusion -->|"all required gates pass"| Supported["Supported"]
    Conclusion -->|"a required gate fails"| Rejected["Not supported"]
    Conclusion -->|"evidence invalid or missing"| Unavailable["Evidence unavailable"]
    Supported --> Next["10. Compare cost or begin a small canary"]
    Rejected --> Repair["10. Diagnose failed cases or stop"]
    Unavailable --> Gather["10. Build or repair the evidence suite"]
```

The scorer changes with the job; the flow does not.

## Candidate Questions, in Learning Order

| Candidate | Its own evaluation question | Baseline | Primary metric |
| --- | --- | --- | --- |
| Untouched baseline | What does Riverside receive before adaptation? | None; this establishes the control record | Contract, prose, latency, and cost measurements required by later claims |
| Continued-pretraining candidate | Does independent Riverside prose become less surprising without unacceptable control-corpus regression? | Untouched base | Token-weighted NLL/perplexity on domain and control corpora |
| SFT candidate | Does the assistant complete the full editor contract more often? | Untouched instruction model or accepted assistant | Complete-contract pass rate plus source-support and safety failures |
| DPO candidate | Do qualified editors prefer it while the accepted SFT contract remains intact? | Accepted SFT candidate | Blinded wins/losses/ties after contract filtering |
| Parameter-strategy candidate | Does a cheaper update preserve the same accepted behavior under a matched experiment? | Accepted behavior-equivalent strategy | Same quality gates first, then measured memory, artifact size, latency, and cost |

Only after these independent evaluations may Riverside compare surviving candidates for the same job. A prose perplexity number must not rank an SFT or DPO assistant, and a preference win rate must not decide whether CPT improved prose modeling.

---

## Setup: Reconstruct Immutable Candidates

Parts 1 and 2 saved independent artifacts under `checkpoints/llm-finetuning/<profile>/`. The setup cells reload fresh model objects so one adapter cannot mutate another candidate's base.

> **Prerequisite:** rerun Parts 1 and 2 from clean kernels on the same hardware profile before loading candidates here.

## One Riverside Request Becomes an Evaluation

Before carrying the formal vocabulary, walk one editing request through the complete chain:

```text
Editor request
"Continue this supplied scene in one sentence, use only its facts, and stop."
        ↓
Baseline and SFT candidate answer the same request
        ↓
Scorer checks: one sentence? clean stop? source-supported? safe?
        ↓
Case record preserves each pass/fail and the failure reasons
        ↓
Metric asks: what fraction of all requests passed every rule?
        ↓
Gate asks: did the precommitted requirement pass?
        ↓
Conclusion: supported, not supported, or evidence unavailable
```

In that one example:

| Formal term | Concrete Riverside object |
| --- | --- |
| Baseline | The untouched instruction model answering the request |
| Candidate | The immutable SFT adapter answering the same request |
| Case | The request, supplied scene, expected constraints, and scorer instructions |
| Scorer | The four checks applied to one response |
| Record | One saved row containing those checks and any failure reasons |
| Metric | Complete-contract pass rate across the versioned request suite |
| Gate | The pass-rate and safety requirements fixed before candidate outputs are inspected |

The rest of Part 3 repeats this shape. CPT changes the scorer to token surprise; DPO changes it to qualified blinded preference; parameter-strategy evaluation keeps the behavior scorer and adds measured resource cost.

### Candidate Manifest: What Will Be Reloaded?

The notebook needs six independent model objects so that loading one adapter cannot mutate another candidate's base model.

| Candidate | Saved artifact | Objective | Parameter strategy | Ancestry |
| --- | --- | --- | --- | --- |
| Baseline | Hugging Face base checkpoint | Original pretraining | No Riverside update | SmolLM2 base |
| Full-FT continuation | `non-instruction-full` | Continued pretraining | Full fine-tuning | SmolLM2 base |
| Partial-freeze continuation | `partial-freeze` | Continued pretraining | Selected late layers | SmolLM2 base |
| LoRA continuation | `peft-lora` | Continued pretraining | LoRA | Fresh SmolLM2 base + adapter |
| SFT assistant | `instruction-lora` | SFT | LoRA | Fresh SmolLM2 base + adapter |
| DPO assistant | `preference-dpo` | DPO | Continue SFT LoRA | Fresh SmolLM2 base + DPO adapter |

The code first restores the common tokenizer and prompt contract, then loads a fresh base for every PEFT adapter, verifies the LoRA target modules, and derives parameter counts from the objects actually loaded. This reconstructs the experiment before any comparison begins.

> **PyTorch → Keras:** `torch.cuda.is_available()` + `.to(device)` explicitly move a model/tensors to
> GPU or CPU, `AutoModelForCausalLM.from_pretrained(...)` loads pretrained weights, and
> `model.generate(...)` run inside `torch.no_grad()` performs autoregressive decoding without tracking
> gradients (nothing to backprop through during inference). **Keras/TF equivalent:** TensorFlow places
> ops on GPU automatically (explicit placement is `tf.device(...)`, rarely needed); the loading call
> would be `TFAutoModelForCausalLM.from_pretrained(...)` followed by the same `.generate(...)` method --
> Keras/TF has no separate "no_grad" context since inference doesn't build a gradient tape by default.


In [ ]:
# Re-establish Parts 1-2's hardware-aware SmolLM2 profile and chat contract.
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_MEMORY_GIB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3 if CUDA_AVAILABLE else 0.0
)

if not CUDA_AVAILABLE:
    MODEL_PROFILE = "cpu-small-135m"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
    INSTRUCT_MODEL_REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"
elif GPU_MEMORY_GIB < 64:
    MODEL_PROFILE = "gpu-balanced-360m"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
    INSTRUCT_MODEL_REVISION = "a10cc1512eabd3dde888204e902eca88bddb4951"
else:
    MODEL_PROFILE = "gpu-quality-1.7b"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    INSTRUCT_MODEL_REVISION = "31b70e2e869a7173562077fd711b654946d38674"

MODEL_NAME = INSTRUCT_MODEL_NAME
MODEL_REVISION = INSTRUCT_MODEL_REVISION
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
SYSTEM_PROMPT = "You are Riverside House's concise fiction-writing assistant."
CONTINUATION_INSTRUCTION = "Continue the fiction narrative in the same style."


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "learning" / "genai" / "content").is_dir() and (candidate / "checkpoints").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from the repository or a descendant directory; "
        "expected learning/genai/content and checkpoints at the repository root."
    )


REPO_ROOT = find_repo_root()
CHECKPOINT_ROOT = REPO_ROOT / "checkpoints"
CHECKPOINT_DIR = CHECKPOINT_ROOT / "llm-finetuning" / MODEL_PROFILE

device = "cuda" if CUDA_AVAILABLE else "cpu"
print(f"Using device: {device} ({MODEL_PROFILE}, {GPU_MEMORY_GIB:.1f} GiB CUDA memory)")
print(f"Profile checkpoints: {CHECKPOINT_DIR}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
).to(device)
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"

num_hidden_layers = len(base_model.model.layers)
hidden_size = base_model.config.hidden_size
total_base_parameters = sum(parameter.numel() for parameter in base_model.parameters())
print(
    f"Loaded {MODEL_NAME}@{MODEL_REVISION[:8]}: {total_base_parameters:,} parameters, "
    f"{num_hidden_layers} decoder layers, hidden size {hidden_size}."
)
if not CUDA_AVAILABLE:
    print(
        "CPU disclaimer: the 135M candidates may all remain generic or visibly similar after "
        "short teaching runs. Treat that as an expected capacity-and-budget result; compare "
        "reserved metrics, artifact size, and parameter cost without claiming production quality."
    )


def instruction_prompt(prompt):
    """Build the user request used by the SFT and DPO recipes."""
    return f"{CONTINUATION_INSTRUCTION}\n\nContext:\n{prompt}"


def apply_instruction_template(prompt):
    """Serialize a request through SmolLM2's native chat template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt.strip()},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate(model, prompt, max_new_tokens=60, use_instruction_template=False):
    """Generate only new tokens, formatting instruction candidates consistently."""
    model.eval()
    model_input = apply_instruction_template(prompt) if use_instruction_template else prompt
    model_device = next(model.parameters()).device
    inputs = tokenizer(model_input, return_tensors="pt")
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(
        output_ids[0][prompt_length:], skip_special_tokens=True
    ).strip()
    return completion if completion else "[model stopped immediately after the prompt]"


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")

### Reloading the Five Fine-Tuned Checkpoints

Each PEFT-wrapped adapter (instruction-tuned LoRA, DPO, LoRA continued pretraining) gets its own fresh
base-model instance rather than sharing `base_model` above -- the same "every PEFT wrapper gets its
own base" rule Parts 1-2 followed throughout. `freeze_model`'s `requires_grad` flags are re-applied
after loading (see the comment below) since that bookkeeping isn't part of a saved checkpoint -- only
the trained weights are.


> **PyTorch → Keras:** `PeftModel.from_pretrained(base_model, path)` wraps a fresh base model with a
> saved LoRA adapter's weights; `named_parameters()` iterates `(name, tensor)` pairs so `requires_grad`
> can be toggled per-parameter (used here to re-apply the freeze pattern, since that bookkeeping isn't
> part of a saved checkpoint), and `p.numel()` counts a tensor's elements to total trainable params.
> **Keras/TF equivalent:** LoRA loading has no single standard TF API (usually a custom `tf.keras.Model`
> subclass or a TF-specific PEFT integration); freezing is coarser-grained -- `layer.trainable = False`
> per layer rather than per-parameter -- and element counts come from `tf.size(variable)`.


In [ ]:
# Reload only artifacts regenerated by Parts 1-2 for MODEL_NAME.

required_checkpoint_artifacts = {
    CHECKPOINT_DIR / "non-instruction-full": ("config.json", "model.safetensors"),
    CHECKPOINT_DIR / "instruction-lora": ("adapter_config.json", "adapter_model.safetensors"),
    CHECKPOINT_DIR / "preference-dpo": ("adapter_config.json", "adapter_model.safetensors"),
    CHECKPOINT_DIR / "partial-freeze": ("config.json", "model.safetensors"),
    CHECKPOINT_DIR / "peft-lora": ("adapter_config.json", "adapter_model.safetensors"),
}
missing_checkpoint_artifacts = [
    checkpoint_dir / artifact_name
    for checkpoint_dir, artifact_names in required_checkpoint_artifacts.items()
    for artifact_name in artifact_names
    if not (checkpoint_dir / artifact_name).is_file()
]
if missing_checkpoint_artifacts:
    missing_lines = "\n".join(f"  - {path}" for path in missing_checkpoint_artifacts)
    raise FileNotFoundError(
        f"Missing required fine-tuning artifacts:\n{missing_lines}\n"
        "Run Parts 1 and 2 first to generate these checkpoints."
    )

non_instruct_ckpt = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR / "non-instruction-full"
).to(device)

instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, CHECKPOINT_DIR / "instruction-lora"
).to(device)

dpo_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
policy_model = PeftModel.from_pretrained(
    dpo_base_reload, CHECKPOINT_DIR / "preference-dpo"
).to(device)

freeze_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR / "partial-freeze"
).to(device)
n_layers = len(freeze_model.model.layers)
unfreeze_from = n_layers - max(2, n_layers // 4)

for parameter in freeze_model.parameters():
    parameter.requires_grad = False
for layer in freeze_model.model.layers[unfreeze_from:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True
for parameter in freeze_model.model.norm.parameters():
    parameter.requires_grad = True

# Llama ties lm_head to token embeddings; keep both frozen in the partial strategy.
assert freeze_model.lm_head.weight is freeze_model.model.embed_tokens.weight
assert not freeze_model.model.embed_tokens.weight.requires_grad

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = PeftModel.from_pretrained(
    lora_pt_base, CHECKPOINT_DIR / "peft-lora"
).to(device)

adapter_models = {
    "instruction LoRA": instruct_lora_model,
    "DPO policy": policy_model,
    "continued-pretraining LoRA": lora_pt_model,
}
expected_targets = set(LORA_TARGET_MODULES)
for adapter_name, adapter_model in adapter_models.items():
    configured_targets = {
        target
        for peft_config in adapter_model.peft_config.values()
        for target in peft_config.target_modules
    }
    if configured_targets != expected_targets:
        raise ValueError(
            f"{adapter_name} targets {sorted(configured_targets)}, expected "
            f"{sorted(expected_targets)}. Rerun Parts 1-2 with the SmolLM2 recipe."
        )

for model in (
    non_instruct_ckpt,
    instruct_lora_model,
    policy_model,
    freeze_model,
    lora_pt_model,
):
    model.eval()

trainable_partial_names = [
    name for name, parameter in freeze_model.named_parameters() if parameter.requires_grad
]
assert any(
    name.startswith(f"model.layers.{unfreeze_from}.")
    for name in trainable_partial_names
), "Expected SmolLM2 trailing-layer parameters were not found"

print("Reloaded all six candidates (baseline + 5 fine-tuned).")
print(f"SmolLM2 layers/hidden size: {n_layers} / {freeze_model.config.hidden_size}")

## Candidate Evaluation 1: Baseline and Continued Pretraining

### Job and Claim

**Baseline job:** establish how surprising Riverside and control prose are before adaptation.

**CPT claim:** on independent corpora, the full-CPT candidate reduces token-weighted Riverside NLL relative to the untouched baseline without exceeding a precommitted control-corpus regression limit.

### Apply the Evaluation Flow

1. **Cases:** independent Riverside passages and an independent general-language control corpus.
2. **Scorer:** teacher-forced actual-token log-probability.
3. **Metric:** token-weighted mean NLL and perplexity, reported separately for domain and control corpora.
4. **Gate:** required domain improvement plus maximum allowed control regression.
5. **Conclusion:** supported, not supported, or evidence unavailable.

The available Aria text is contaminated, so the cells below demonstrate the scorer and aggregation but cannot apply a production gate.

### Give Every Model the Same Exam Sheet

A generated sample is the model writing its own exam answer. Once two models choose different tokens, every later prediction receives different context, so the outputs no longer isolate the model change. Perplexity evaluation instead supplies one fixed held-out passage as the answer key. The base and adapted models read the same preceding words and are scored against the same actual next token at every position.

At each position, softmax gives the model one probability budget to distribute across its vocabulary. We record how much of that budget reached the token that actually occurred. A high value means the text was expected; a low value means most probability went elsewhere. That one value does **not** reveal whether the remaining mass sat on one plausible alternative or thousands of weak ones, but it is exactly the quantity the reference text lets us score consistently.

> The models do not write the exam. They assign probabilities to the same answer key.

### Teacher Forcing: Follow the Actual Text

Use one fixed Riverside continuation:

```text
Aria Voss checked the Meridian's Promise status panel
and opened the Keeper's maintenance logs.
```

Both models take the same token-by-token exam:

| Step | True prefix supplied to both models | Actual next token scored |
| ---: | --- | --- |
| 1 | `... status panel and` | ` opened` |
| 2 | `... status panel and opened` | ` the` |
| 3 | `... status panel and opened the` | the next tokenizer piece |

At each step, inspect the model's full next-token distribution but record the probability assigned to the token that actually occurred. Then append that **actual token** to the prefix, regardless of what either model would have generated, and continue.

This is **teacher forcing**. It keeps the histories from forking, so a difference in probability belongs to the models rather than to an earlier sampling decision.

The interpretation remains narrow: high actual-token probability means “expected under this model after this context,” not true, safe, or editorially preferred.

### From Actual-Token Probability to Mean Surprise

For a two-token illustration, suppose one model assigns `50%` probability to ` opened`, then `20%` probability to ` the` after receiving the true prefix.

| Actual token | Probability budget reaching it | Reading |
| --- | ---: | --- |
| ` opened` | `50%` | The model kept several alternatives plausible |
| ` the` | `20%` | Most probability went to other possible next tokens |

A complete continuation may contain many tokenizer pieces. Multiplying all their probabilities quickly produces an inconveniently tiny sequence number, so implementations convert each actual-token probability into **surprise** on a log scale. Predictable tokens contribute little surprise; tokens the model nearly ruled out contribute a large penalty.

Average those penalties across the continuation and you have **mean negative log-likelihood (mean NLL)**: the model's average surprise per actual token. Lower mean NLL means the fixed text was more expected.

The next code cell performs the exact calculation for every tokenizer piece under both models. The continuation is reference text supplied for scoring; the metric does not claim it is the uniquely correct continuation.

### Compare the Model Changes

A mean log-probability is one score for **one model reading one fixed continuation**. The baseline comparison appears only when we put the same continuation under both models.

The values below are illustrative. The code calculates the same columns from the real checkpoints.

| Fixed continuation after the same prompt | Base model score | Adapted model score | Change after adaptation |
| --- | ---: | ---: | ---: |
| Riverside target: ` opened the Keeper's maintenance logs` | $-5.0$ | $-3.0$ | $+2.0$ |
| Generic control: ` looked at the screen` | $-2.0$ | $-1.8$ | $+0.2$ |

Read the matrix in this order:

1. **Stay within one row first.** For the Riverside phrase, the score moved from $-5.0$ to $-3.0$, so this fixed phrase became less surprising after adaptation.
2. **Repeat the same before/after comparison for the generic phrase.** It also became less surprising, but only by $+0.2$.
3. **Compare the two changes, not the raw scores.** The Riverside phrase gained $+2.0$, while the generic phrase gained $+0.2$. The Riverside phrase therefore gained $+1.8$ more.

Do **not** compare the raw adapted scores $-3.0$ and $-1.8$ directly. The generic phrase may simply have been easier for both models before training. The useful question is whether adaptation changed each phrase differently from its own starting point.

The short labels for those two subtractions are:

- phrase shift: $\Delta(c)=\text{adapted score for }c-\text{base score for }c$;
- selectivity contrast: $\Delta(\text{Riverside phrase})-\Delta(\text{generic phrase})$.

For the worked matrix, the selectivity contrast is $(+2.0)-(+0.2)=+1.8$. For this pair, the positive result suggests that adaptation favored the Riverside-specific phrase more than the plausible generic alternative.

The **Riverside target** is a deliberately domain-specific probe phrase, not “the one correct answer.” The **generic control** is a plausible continuation with no Riverside-specific content. It tells us whether the score change was broad rather than specifically tied to the Riverside material.

This still covers only the chosen phrases. It does not establish generalization, overall writing quality, factual correctness, or instruction following.

### Make the Riverside Prediction Before Measuring

**Observation:** the adapted model produced Riverside-sounding text, but sampling may have made it look better by chance.

**Prediction:** the full-CPT candidate should raise support for an Aria-specific continuation more than for a plausible generic continuation when both are compared with their own untouched-baseline scores.

**Evidence:** hold the words fixed, score them under both models, and compare each phrase with its own starting point. Repeating the contract on a second story shows that the method is reusable; neither scene is clean production evidence.

> **PyTorch → Keras:** `model.eval()` switches dropout/batch-norm-style layers to inference behavior;
> `torch.no_grad()` disables gradient tracking for the forward pass below; calling `model(**inputs)`
> runs a forward pass and returns `outputs.logits` (raw scores), and `F.softmax(logits, dim=-1)`
> (from `torch.nn.functional`) converts those logits into a probability distribution over the vocabulary.
> **Keras/TF equivalent:** Keras layers infer train/inference behavior automatically (or via a
> `training=False` argument) instead of an explicit `.eval()` call, and there's no separate "no_grad"
> context since plain forward calls outside a `GradientTape` don't track gradients; the softmax step is
> `tf.nn.softmax(logits, axis=-1)` -- same idea, `axis` instead of `dim`.


In [ ]:
# Trace one fixed sentence token by token, then compare complete continuation scores.
import math

import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

prompt_for_analysis = "Aria Voss checked the Meridian's Promise status panel and"
target_phrase_label = "Keeper's maintenance logs"
generic_control_label = "looked at the screen"

# The first phrase is the target; the remaining phrases test selectivity.
candidate_phrases = {
    target_phrase_label: " opened the Keeper's maintenance logs",
    "quantum fold drive": " checked the quantum fold drive",
    "containment-field anomaly": " detected a containment-field anomaly",
    generic_control_label: " looked at the screen",
    "said nothing": " said nothing",
    "went back to work": " went back to work",
}
domain_labels = set(list(candidate_phrases)[:3])


def continuation_logprob(model, prompt, continuation):
    """Return sequence summaries and the actual-token trace for one continuation."""
    prompt_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = tokenizer(
        prompt + continuation, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    prompt_length = prompt_ids.shape[1]
    if not torch.equal(full_ids[:, :prompt_length], prompt_ids):
        raise ValueError("Prompt tokens are not a stable prefix of prompt + continuation")

    model.eval()
    with torch.no_grad():
        logits = model(full_ids).logits[:, :-1, :]
        log_probs = F.log_softmax(logits, dim=-1)

    # Position prompt_length-1 predicts the first continuation token.
    continuation_ids = full_ids[:, prompt_length:]
    continuation_log_probs = log_probs[
        0, prompt_length - 1 : full_ids.shape[1] - 1
    ].gather(1, continuation_ids[0].unsqueeze(1)).squeeze(1)

    token_ids = continuation_ids[0].tolist()
    token_log_probs = continuation_log_probs.tolist()
    token_pieces = tokenizer.convert_ids_to_tokens(token_ids)
    trace = [
        {
            "token_id": token_id,
            "token": token_piece,
            "probability": math.exp(token_log_probability),
            "log_probability": token_log_probability,
            "surprise": -token_log_probability,
        }
        for token_id, token_piece, token_log_probability in zip(
            token_ids, token_pieces, token_log_probs
        )
    ]

    return {
        "mean": continuation_log_probs.mean().item(),
        "sum": continuation_log_probs.sum().item(),
        "tokens": continuation_ids.shape[1],
        "trace": trace,
    }


phrase_results = []
target_traces = None
for label, phrase in candidate_phrases.items():
    baseline_score = continuation_logprob(base_model, prompt_for_analysis, phrase)
    finetuned_score = continuation_logprob(
        non_instruct_ckpt, prompt_for_analysis, phrase
    )
    if label == target_phrase_label:
        target_traces = {
            "baseline": baseline_score["trace"],
            "finetuned": finetuned_score["trace"],
        }
    phrase_results.append(
        {
            "label": label,
            "kind": "catalog" if label in domain_labels else "generic control",
            "tokens": finetuned_score["tokens"],
            "baseline": baseline_score["mean"],
            "finetuned": finetuned_score["mean"],
            "delta": finetuned_score["mean"] - baseline_score["mean"],
        }
    )

assert target_traces is not None
assert [item["token_id"] for item in target_traces["baseline"]] == [
    item["token_id"] for item in target_traces["finetuned"]
]

print("=== Fixed target: actual next-token trace ===")
print(f"Prompt: {prompt_for_analysis!r}")
print(f"Fixed target: {candidate_phrases[target_phrase_label]!r}\n")
print(
    f"{'Step':>4} {'Tokenizer piece':22} {'Base p':>10} {'Adapted p':>10} "
    f"{'Base surprise':>14} {'Adapted surprise':>17}"
)
print("-" * 92)
for step, (baseline_token, finetuned_token) in enumerate(
    zip(target_traces["baseline"], target_traces["finetuned"]), start=1
):
    print(
        f"{step:>4} {baseline_token['token']!r:22} "
        f"{baseline_token['probability']:>10.5f} "
        f"{finetuned_token['probability']:>10.5f} "
        f"{baseline_token['surprise']:>14.3f} "
        f"{finetuned_token['surprise']:>17.3f}"
    )
print(
    "\nRead one row at a time: both models see the same true prefix; lower surprise "
    "means the model assigned more probability to that actual next token."
)

print("\n=== Complete fixed-continuation scores ===")
print(
    f"{'Phrase':30} {'Type':16} {'Tok':>3} {'Base':>9} "
    f"{'Adapted':>11} {'Adapted - base':>15}"
)
print("-" * 94)
for row in phrase_results:
    print(
        f"{row['label']:30} {row['kind']:16} {row['tokens']:>3} "
        f"{row['baseline']:>9.3f} {row['finetuned']:>11.3f} {row['delta']:>+15.3f}"
    )

labels = [row["label"] for row in phrase_results]
baseline_values = [row["baseline"] for row in phrase_results]
finetuned_values = [row["finetuned"] for row in phrase_results]
deltas = [row["delta"] for row in phrase_results]
positions = np.arange(len(labels))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.35, 1]})

axes[0].barh(
    positions - width / 2,
    baseline_values,
    height=width,
    label="Baseline",
    color="#4C78A8",
)
axes[0].barh(
    positions + width / 2,
    finetuned_values,
    height=width,
    label="Continued pretraining",
    color="#E45756",
)
axes[0].set_yticks(positions)
axes[0].set_yticklabels(labels)
axes[0].invert_yaxis()
axes[0].set_xlabel("Mean log-probability per token (higher = less surprising)")
axes[0].set_title("One-Model Scores for Fixed Text")
axes[0].legend()
axes[0].grid(axis="x", alpha=0.25)

delta_colors = ["#2A9D8F" if value >= 0 else "#D1495B" for value in deltas]
axes[1].barh(positions, deltas, color=delta_colors)
axes[1].set_yticks(positions)
axes[1].set_yticklabels(labels)
axes[1].invert_yaxis()
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("Adapted minus base (nats/token)")
axes[1].set_title("Baseline Comparison for Each Phrase")
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

catalog_deltas = [row["delta"] for row in phrase_results if row["kind"] == "catalog"]
control_deltas = [
    row["delta"] for row in phrase_results if row["kind"] == "generic control"
]
target_delta = next(
    row["delta"] for row in phrase_results if row["label"] == target_phrase_label
)
generic_control_delta = next(
    row["delta"] for row in phrase_results if row["label"] == generic_control_label
)
target_selectivity_delta = target_delta - generic_control_delta

print("\n=== Nested comparisons ===")
print(f"Target phrase, adapted - base:          {target_delta:+.3f} nats/token")
print(
    f"Named generic control, adapted - base: {generic_control_delta:+.3f} nats/token"
)
print(
    f"Selectivity contrast (target - control): {target_selectivity_delta:+.3f} nats/token"
)
print(f"Mean catalog shift:                    {np.mean(catalog_deltas):+.3f} nats/token")
print(f"Mean all-generic-controls shift:        {np.mean(control_deltas):+.3f} nats/token")
print(
    "Interpretation: each phrase shift compares adapted with base. The selectivity "
    "contrast then asks whether the target phrase shifted more than the named generic control."
)
print(
    "A positive selectivity contrast suggests a local Riverside-specific pattern; "
    "it is not a general quality or corpus-level result."
)

print("\nCPT CANDIDATE DIAGNOSTIC")
print(f"Aria selectivity contrast: {target_selectivity_delta:+.3f} nats/token")
print(
    "This diagnostic belongs only to the full-CPT candidate and its untouched base. "
    "The same calculation must be repeated on independent Riverside and control corpora "
    "before a CPT claim can be supported."
)

### Broaden the Evidence: From One Token to Two Corpora

One fixed continuation is a useful microscope, not a product case. Carry the same scoring contract outward without changing the question:

```text
same held-out text
-> probability on each actual next token
-> surprise for each actual token
-> token-weighted mean surprise (mean NLL)
-> perplexity for reporting
-> compare base and adapted models
```

Run that path separately on two independent corpora:

| Corpus | What improvement would mean | What regression would warn about |
| --- | --- | --- |
| Riverside domain text | The adapted model finds Riverside prose less surprising | No domain gain means CPT did not transfer to unseen Riverside text |
| General-language control text | Stable scores suggest broad language modeling was preserved | Higher surprise may indicate forgetting or over-specialization |

The same-corpus probe below demonstrates the calculation but remains contaminated by training exposure. A real gate needs separately versioned text that was never used to create training examples or select checkpoints.

### What Perplexity Adds Beyond Mean NLL

Mean NLL already contains the complete scoring information. Perplexity does **not** add another observation or inspect additional tokens; it translates that log-scaled average into a more interpretable scale.

| Mean NLL | Perplexity | Uniform-equivalent reading |
| ---: | ---: | --- |
| `0.00` | `1.0` | The actual token was effectively certain at every step |
| `0.69` | `2.0` | Uncertainty comparable to a uniform choice between 2 options |
| `2.30` | `10.0` | Uncertainty comparable to a uniform choice among 10 options |
| `4.61` | `100.0` | Uncertainty comparable to a uniform choice among 100 options |

This is an **equivalent choice count**, not a literal count of contenders. Perplexity `10` does not prove the model considered exactly ten tokens equally likely; it says the average surprise matches that hypothetical uniform choice.

Read the two metrics as two views of the same result:

```text
mean NLL   -> precise log-scale value used by the computation
perplexity -> the same value translated for human interpretation
```

Lower perplexity means the model expected the supplied text more strongly. It does not establish truth, reasoning, instruction following, safety, or editorial quality. Raw values also depend on tokenization and context length, so compare candidates under the same tokenizer, text, boundaries, and scoring procedure.

### Repeat the Check Across More Passages

The selected continuation may be unusually favorable. Repeat the same teacher-forced scoring across many Riverside passages:

> Does the adapted model generally find the words that actually occur less surprising than the base model does?

This notebook's shared corpus check is useful for seeing how the calculation scales up. Some candidates may already have seen parts of that text, so do not use its bars to choose a production model.

> **PyTorch → Keras:** passing `labels=enc["input_ids"]` into the model's forward call makes the
> Hugging Face model compute cross-entropy loss internally and return it as `out.loss` (a scalar
> tensor); `torch.no_grad()` skips gradient tracking since this is evaluation-only, and `.item()`
> pulls the plain Python float out of that 0-d tensor, which `math.exp(loss)` then turns into
> perplexity. **Keras/TF equivalent:** the TF counterpart model supports the same `labels=` convenience
> (`model(enc, labels=...)`), while plain Keras code would instead call
> `tf.keras.losses.SparseCategoricalCrossentropy()(y_true, logits)` and use `.numpy()` in place of
> `.item()` to extract the scalar.


In [ ]:
# Reuse the repository path resolved during setup and load the GenAI-wide corpus.
CONTENT_DIR = REPO_ROOT / "learning" / "genai" / "content"

NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}

if not CONTENT_DIR.exists():
    raise FileNotFoundError(
        f"Shared Riverside content directory not found at {CONTENT_DIR.absolute()}. "
        "Expected the GenAI-wide corpus under learning/genai/content."
    )

print(f"Corpus evaluation setup: {CONTENT_DIR.absolute()} ({len(NOVELS)} novels)")


In [ ]:
import math


# Use the same later-chapter sample for every candidate. This is a shared probe, not a clean holdout,
# because upstream training coverage differs and full FT saw some of these files.
def load_corpus_probe(start_chapter_index=10, chapters_per_novel=2, min_len=200):
    paragraphs = []
    source_files = []

    for alias, novel_dir in NOVELS.items():
        novel_path = CONTENT_DIR / novel_dir
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        probe_files = chapter_files[
            start_chapter_index : start_chapter_index + chapters_per_novel
        ]
        source_files.extend(probe_files)

        for path in probe_files:
            text = path.read_text(encoding="utf-8")
            for paragraph in text.split("\n\n"):
                paragraph = paragraph.strip().replace("\n", " ")
                if len(paragraph) >= min_len:
                    paragraphs.append(paragraph)

    return paragraphs, source_files


probe_paragraphs, probe_files = load_corpus_probe()
print(
    f"Shared corpus probe: {len(probe_paragraphs)} paragraphs from "
    f"{len(probe_files)} later-chapter files."
)
print(
    "WARNING: this is descriptive, not held out. Upstream candidates used different "
    "training files, and full fine-tuning saw some probe chapters."
)


# Aggregate negative log-likelihood by evaluated token rather than averaging paragraph means.
def compute_corpus_probe(model, paragraphs, max_length=64):
    model.eval()
    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        for paragraph in paragraphs:
            encoded = tokenizer(
                paragraph,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            output = model(**encoded, labels=encoded["input_ids"])

            # Causal loss predicts tokens 2..N from tokens 1..N-1.
            valid_tokens = int(encoded["attention_mask"][:, 1:].sum().item())
            total_nll += output.loss.item() * valid_tokens
            total_tokens += valid_tokens

    mean_nll = total_nll / total_tokens
    return {
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "tokens": total_tokens,
    }


models_for_probe = {
    "Untouched baseline": base_model,
    "Full-CPT candidate": non_instruct_ckpt,
}

print(f"\n{'=' * 72}\nShared corpus-probe perplexity (descriptive only):\n{'=' * 72}")
corpus_probe_results = {}
for name, model in models_for_probe.items():
    result = compute_corpus_probe(model, probe_paragraphs)
    corpus_probe_results[name] = result
    print(
        f"  {name:<32} NLL={result['mean_nll']:6.3f}  "
        f"PPL={result['perplexity']:8.1f}  tokens={result['tokens']:,}"
    )
print(f"{'=' * 72}")

baseline_probe = corpus_probe_results["Untouched baseline"]
cpt_probe = corpus_probe_results["Full-CPT candidate"]
print(
    f"\nDescriptive CPT change: NLL={cpt_probe['mean_nll'] - baseline_probe['mean_nll']:+.3f}, "
    f"PPL={cpt_probe['perplexity'] - baseline_probe['perplexity']:+.1f}."
)
print(
    "Conclusion: evidence unavailable for a generalization claim because this probe overlaps "
    "the training corpus. Repeat the same baseline-versus-CPT calculation on independent "
    "Riverside and control corpora before applying a gate."
)

## Candidate Evaluation 2: SFT Assistant

### Job and Claim

**Job:** answer Riverside editing requests under a complete response contract.

**Claim:** on an independent editing suite, the SFT candidate increases complete usable-turn rate relative to the untouched instruction baseline while preserving source support and safety.

### Apply the Evaluation Flow

1. **Baseline:** untouched SmolLM2-Instruct under the same chat template.
2. **Cases:** versioned requests containing a source passage and explicit rules such as `one sentence`, `stop after the sentence`, and `use only supplied facts`.
3. **Scorer:** one record per response with `one_sentence`, `clean_stop`, `source_supported`, and `safety_pass`.
4. **Metric:** a case passes only when every required rule passes.

Suppose five cases are evaluated and only two satisfy every rule. The complete-contract pass rate is `2 / 5 = 40%`; a case that passes three rules but fails the fourth still contributes zero complete passes.

The general definition is:

$$
\text{complete-contract pass rate}
=\frac{\text{cases passing every required rule}}{N}.
$$

5. **Gate:** a product-owned pass-rate floor plus zero critical safety failures, fixed before results are inspected.
6. **Conclusion:** supported, not supported, or evidence unavailable according to the vocabulary defined at the top.

### Worked Case Records

The next code cell uses five explicitly synthetic records to demonstrate aggregation. Names such as `SFT-01` are local case IDs, not benchmark examples or measured model outputs.

In [ ]:
# Synthetic SFT records: demonstrate complete-contract aggregation only.
worked_sft_cases = [
    {
        "case_id": "SFT-01",
        "request": "Continue a supplied Aria passage in one sentence and stop.",
        "one_sentence": True,
        "clean_stop": True,
        "source_supported": True,
        "safety_pass": True,
    },
    {
        "case_id": "SFT-02",
        "request": "Return one sentence, but the response contains two sentences.",
        "one_sentence": False,
        "clean_stop": True,
        "source_supported": True,
        "safety_pass": True,
    },
    {
        "case_id": "SFT-03",
        "request": "Use only the supplied passage, but the response invents a detail.",
        "one_sentence": True,
        "clean_stop": True,
        "source_supported": False,
        "safety_pass": True,
    },
    {
        "case_id": "SFT-04",
        "request": "Stop after one sentence, but the response adds another turn.",
        "one_sentence": True,
        "clean_stop": False,
        "source_supported": True,
        "safety_pass": True,
    },
    {
        "case_id": "SFT-05",
        "request": "Continue a second supplied passage in one sentence and stop.",
        "one_sentence": True,
        "clean_stop": True,
        "source_supported": True,
        "safety_pass": True,
    },
]
required_rules = ("one_sentence", "clean_stop", "source_supported", "safety_pass")

print("SYNTHETIC SFT CONTRACT RECORDS")
print(f"{'ID':8} {'sentence':>9} {'stop':>7} {'support':>9} {'safety':>8} {'pass':>7}")
print("-" * 58)
for record in worked_sft_cases:
    record["contract_pass"] = all(record[rule] for rule in required_rules)
    print(
        f"{record['case_id']:8} "
        f"{str(record['one_sentence']):>9} "
        f"{str(record['clean_stop']):>7} "
        f"{str(record['source_supported']):>9} "
        f"{str(record['safety_pass']):>8} "
        f"{str(record['contract_pass']):>7}"
    )

complete_passes = sum(record["contract_pass"] for record in worked_sft_cases)
instruction_pass_rate = complete_passes / len(worked_sft_cases)
print(
    f"\nComplete-contract pass rate = {complete_passes} / "
    f"{len(worked_sft_cases)} = {instruction_pass_rate:.1%}"
)
print("These records demonstrate aggregation; they are not measured candidate results.")

### Do Not Let the Average Hide the Failure

A pass rate answers **how often** the contract held. It does not answer **which obligation broke**, **where failures cluster**, or **how uncertain the estimate is**.

Riverside therefore keeps three views of the same run:

1. **Case records:** the response, scorer outputs, and failure reasons for every request.
2. **Slices:** pass rate and failure counts by request type, language, length, risk tier, and other product-owned groups.
3. **Uncertainty:** an interval around the aggregate, paired with a precommitted minimum sample size.

The next cell reuses the five synthetic records. Its wide bootstrap interval is the lesson: five examples can demonstrate aggregation code, but cannot support a production gate. Bootstrap resampling also cannot repair an unrepresentative suite or a miscalibrated scorer.

In [ ]:
# Synthetic diagnostic exercise: failure attribution and uncertainty, not candidate evidence.
from collections import Counter
import random

failure_counts = Counter()
for record in worked_sft_cases:
    failed_rules = [rule for rule in required_rules if not record[rule]]
    record["failure_reasons"] = failed_rules
    failure_counts.update(failed_rules)

observed_passes = [int(record["contract_pass"]) for record in worked_sft_cases]
bootstrap_rng = random.Random(2026)
bootstrap_rates = []
for _ in range(10_000):
    resample = [bootstrap_rng.choice(observed_passes) for _ in observed_passes]
    bootstrap_rates.append(sum(resample) / len(resample))
bootstrap_rates.sort()
low_index = int(0.025 * len(bootstrap_rates))
high_index = int(0.975 * len(bootstrap_rates)) - 1
bootstrap_interval = (bootstrap_rates[low_index], bootstrap_rates[high_index])

print("SYNTHETIC FAILURE ATTRIBUTION")
for rule in required_rules:
    print(f"  {rule:<18} {failure_counts[rule]} failures")
print("\nCASE-LEVEL DIAGNOSIS")
for record in worked_sft_cases:
    reasons = ", ".join(record["failure_reasons"]) or "none"
    print(f"  {record['case_id']}: {reasons}")
print(
    f"\nObserved pass rate: {instruction_pass_rate:.1%}; "
    f"illustrative 95% bootstrap interval: "
    f"[{bootstrap_interval[0]:.1%}, {bootstrap_interval[1]:.1%}]"
)
print("Decision: demonstration only. The suite is too small and synthetic for a release gate.")

## Candidate Evaluation 3: DPO Assistant

### Job and Claim

**Job:** improve editorial usefulness among answers that already satisfy the accepted SFT contract.

**Claim:** on independent matched requests, qualified blinded reviewers prefer DPO to the accepted SFT candidate while DPO preserves every required contract and safety gate.

### Apply the Evaluation Flow

1. **Baseline:** the accepted SFT artifact, not the untouched model.
2. **Cases:** the same independent requests sent to SFT and DPO with matched decoding settings.
3. **Eligibility filter:** if either response fails the SFT contract or safety requirement, record that failure separately; do not hide it inside preference.
4. **Scorer:** randomize A/B order, hide model identity, give reviewers one written rubric, and record `DPO win`, `SFT win`, or `tie`.
5. **Metric:** preserve wins, losses, and ties separately before computing credited win rate.

Suppose seven eligible comparisons produce three DPO wins, two SFT wins, and two ties. Giving each tie half credit produces `3 + 0.5 x 2 = 4` credited DPO wins, so the credited win rate is `4 / 7 = 57.1%`.

The general rule is:

$$
\text{credited DPO win rate}
=\frac{W+0.5T}{W+L+T}.
$$

6. **Gate:** contract retention first, then a product-owned preference floor and reviewer-quality requirements.
7. **Conclusion:** DPO is supported only if both the contract gate and preference gate pass. Preference cannot rescue a contract regression.

### Worked Judgment Records

The next code cell uses eight synthetic judgments only to show how wins, losses, ties, and a contract-ineligible case aggregate. They are not editor-study results.

In [ ]:
# Synthetic blinded DPO judgments: demonstrate preference aggregation and uncertainty.
import random

worked_preference_judgments = [
    {"case_id": "DPO-01", "contract_eligible": True, "judgment": "DPO win"},
    {"case_id": "DPO-02", "contract_eligible": True, "judgment": "SFT win"},
    {"case_id": "DPO-03", "contract_eligible": True, "judgment": "DPO win"},
    {"case_id": "DPO-04", "contract_eligible": True, "judgment": "tie"},
    {"case_id": "DPO-05", "contract_eligible": True, "judgment": "DPO win"},
    {"case_id": "DPO-06", "contract_eligible": True, "judgment": "tie"},
    {"case_id": "DPO-07", "contract_eligible": True, "judgment": "SFT win"},
    {"case_id": "DPO-08", "contract_eligible": False, "judgment": None},
]

eligible_judgments = [
    record["judgment"]
    for record in worked_preference_judgments
    if record["contract_eligible"]
]
dpo_wins = eligible_judgments.count("DPO win")
sft_wins = eligible_judgments.count("SFT win")
ties = eligible_judgments.count("tie")
dpo_win_rate = (dpo_wins + 0.5 * ties) / len(eligible_judgments)

print("SYNTHETIC BLINDED DPO JUDGMENTS")
for record in worked_preference_judgments:
    result = record["judgment"] if record["contract_eligible"] else "contract failure: excluded"
    print(f"  {record['case_id']}: {result}")
print(
    f"\nCredited DPO win rate = ({dpo_wins} + 0.5 * {ties}) / "
    f"{len(eligible_judgments)} = {dpo_win_rate:.1%}"
)
print(f"SFT wins remain explicit: {sft_wins}")

preference_credits = [
    1.0 if judgment == "DPO win" else 0.5 if judgment == "tie" else 0.0
    for judgment in eligible_judgments
]
preference_rng = random.Random(2027)
preference_bootstrap_rates = []
for _ in range(10_000):
    resample = [preference_rng.choice(preference_credits) for _ in preference_credits]
    preference_bootstrap_rates.append(sum(resample) / len(resample))
preference_bootstrap_rates.sort()
preference_low = preference_bootstrap_rates[int(0.025 * len(preference_bootstrap_rates))]
preference_high = preference_bootstrap_rates[int(0.975 * len(preference_bootstrap_rates)) - 1]
print(
    f"Illustrative 95% bootstrap interval: [{preference_low:.1%}, {preference_high:.1%}]"
)
print("These judgments demonstrate aggregation and uncertainty; they are not editor-study results.")

## Candidate Evaluation 4: Parameter Strategy

### Job and Claim

**Job:** reduce update and operating burden without losing an already accepted behavior.

**Claim:** under a matched experiment, a cheaper parameter strategy preserves the same behavior gates as the accepted strategy and improves at least one measured resource requirement.

### Apply the Evaluation Flow

1. Hold the base revision, ordered data, objective, split, token budget, optimizer policy, seeds, evaluation suite, and decoding fixed.
2. Change only where the update is stored: full weights, selected layers, or LoRA matrices.
3. Apply the same behavior, source-support, and safety gates to every strategy.
4. Eliminate any strategy that fails required behavior.
5. Among survivors, compare measured training memory, training time, artifact bytes, serving latency, and request cost.

The next code cell reports update scope only. Part 2 now holds the Aria corpus, causal objective, and ten-step teaching budget fixed across partial freezing and LoRA, but it does not yet hold every optimizer setting fixed or run an independent behavior suite. These counts therefore do not establish that one strategy caused a quality difference; a causal decision requires the fully matched study above.

In [ ]:
# Descriptive parameter scope: cost context, not quality evidence.
required_models = ("base_model", "non_instruct_ckpt", "freeze_model", "lora_pt_model")
if all(model_name in globals() for model_name in required_models):
    total_params = sum(parameter.numel() for parameter in base_model.parameters())
    strategy_counts = {
        "Full fine-tuning": sum(parameter.numel() for parameter in non_instruct_ckpt.parameters()),
        "Partial freezing": sum(
            parameter.numel() for parameter in freeze_model.parameters() if parameter.requires_grad
        ),
        "LoRA matrices": sum(
            parameter.numel()
            for name, parameter in lora_pt_model.named_parameters()
            if ".lora_" in name
        ),
    }

    print("DESCRIPTIVE UPDATE SCOPE")
    for strategy, count in strategy_counts.items():
        print(f"  {strategy:<20} {count:>12,} parameters ({count / total_params:>7.2%})")
    print(
        "These counts show how much state each strategy can update. They do not show "
        "which strategy preserves behavior; that requires the matched evaluation described above."
    )
else:
    print("Run the candidate-loading cells to inspect update scope.")

## Matched Comparison: Change One Thing, Then Price the Difference

To learn whether LoRA or full fine-tuning caused a difference, hold the base model, ordered examples, split, instruction template, optimizer policy, token budget, evaluation code, and seeds fixed. Change only how the update is stored.

1. Check instruction following, source support, and safety first.
2. Inspect difficult requests instead of trusting only the average.
3. Compare memory, training time, artifact size, latency, and cost only among candidates that preserve the required behavior.

Choose LoRA when it keeps Riverside's workflow and removes a real burden. Choose full fine-tuning only when its repeatable improvement matters enough to pay for.

```mermaid
flowchart TD
    Fixed["Fix base, data, objective, budget, seeds, and suite"] --> Strategies["Train full, partial, and LoRA strategies"]
    Strategies --> Behavior["Apply identical behavior and safety gates"]
    Behavior -->|"fails"| Drop["Eliminate strategy"]
    Behavior -->|"passes"| Cost["Measure memory, time, bytes, latency, and cost"]
    Cost --> Choice["Choose the least burdensome supported strategy"]
```

The existing teaching checkpoints locate the clean experiment Riverside still needs; they do not answer it.

## Compare Survivors, Then Decide

Independent evaluation comes first. Comparison is allowed only after candidates have passed the requirements for the **same job**.

### What May Be Compared

| Situation | Valid comparison |
| --- | --- |
| CPT versus untouched base | Domain/control NLL or perplexity on the same independent corpora |
| SFT versus untouched instruction baseline | Complete-contract, source-support, safety, latency, and cost on the same editing cases |
| DPO versus accepted SFT | Contract retention plus blinded preference on the same matched requests |
| Full versus partial versus LoRA | Same objective, data, seed plan, and suite; behavior gates first, resource cost second |

Do not merge unrelated metrics into one universal score. A lower prose perplexity cannot compensate for a failed instruction contract, and a DPO preference win cannot compensate for a safety failure.

### Decision Table

| Conclusion | Meaning | Riverside action |
| --- | --- | --- |
| **Supported** | Valid required evidence exists and every required gate passes | Record the artifact, suite, code, policy, and metrics; begin a small canary |
| **Not supported** | Valid required evidence exists and at least one required gate fails | Keep the accepted release; inspect failed cases; revise or stop the candidate |
| **Evidence unavailable** | Required evidence is missing, contaminated, or too uncertain | Build or repair the suite; gather judgments; do not promote |

The current teaching artifacts have **evidence unavailable** for production claims because all available Aria prose was used during training. Same-corpus diagnostics can debug the pipeline, but they cannot change that conclusion.

### From Offline Evidence to Release

```mermaid
flowchart TD
    Independent["Independent suite + immutable candidate"] --> Gates["All precommitted offline gates"]
    Gates -->|"fail"| Keep["Keep accepted release and diagnose cases"]
    Gates -->|"evidence invalid or missing"| Repair["Repair suite or gather evidence"]
    Gates -->|"pass"| Record["Write decision and lineage record"]
    Record --> Canary["Begin small monitored canary"]
    Canary -->|"live gate degrades"| Rollback["Route to immutable rollback artifact"]
    Canary -->|"healthy"| Expand["Increase traffic gradually"]
```

A canary is not another offline metric. It tests live prompt mix, latency, cost, and failure modes that the fixed suite may not represent. The deeper [LLM evaluation arc](../04-llm-evaluation/01-llm-evaluation-metrics-and-benchmarks.ipynb) covers evaluator calibration, agreement, uncertainty, difficult slices, and adversarial cases.

### Apply the Guidebook in Production-Style Code

The disabled workflow below enforces the boundary created by full-corpus training:

1. load a separately versioned benchmark result with content-addressed `dataset_files`;
2. reconstruct the training-corpus records from Part 1's artifact manifest;
3. verify the benchmark's declared fingerprint matches its file records;
4. reject the benchmark unless it declares `dataset_role: external_evaluation`, has a different aggregate fingerprint, and shares no file digest with training;
5. apply only the workload gates configured in advance;
6. write a content-addressed decision record; and
7. package an immutable release only when every enabled gate passes.

Same-corpus notebook probes are never copied into this release decision.

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import shutil
from typing import Any, Mapping, Optional


RUN_PRODUCTION_DECISION = False


@dataclass(frozen=True)
class EvaluationPolicy:
    """Illustrative policy shape; replace enabled defaults with product-owned requirements."""

    min_instruction_pass_rate: Optional[float] = 0.95
    min_source_support_rate: Optional[float] = 0.98
    min_preference_win_rate: Optional[float] = None
    min_safety_pass_rate: Optional[float] = 1.0
    max_p95_latency_ms: Optional[float] = 1_500.0
    max_cost_per_1k_requests_usd: Optional[float] = 1.00
    max_perplexity_regression_pct: Optional[float] = None


@dataclass(frozen=True)
class ProductionDecisionConfig:
    workload: str = "editing-assistant"
    candidate_name: str = "Instruction-tuned (LoRA)"
    candidate_artifact: Path = CHECKPOINT_DIR / "instruction-lora"
    candidate_strategy: str = "lora"
    training_manifest: Path = CHECKPOINT_DIR / "instruction-lora" / "experiment-manifest.json"
    rollback_name: str = "previous-production"
    rollback_artifact: Path = Path("./artifacts/production/current")
    benchmark_metrics: Path = Path("./artifacts/production-benchmarks.json")
    registry_dir: Path = Path("./artifacts/finetuning-decisions")
    release_root: Path = Path("./artifacts/production-releases")
    base_model_id: str = MODEL_NAME
    base_revision: str = MODEL_REVISION
    policy: EvaluationPolicy = EvaluationPolicy()


def canonical_dataset_fingerprint(records) -> str:
    """Hash sorted path/content-digest records into one dataset identity."""
    normalized = sorted(
        (
            {"path": record["path"], "sha256": record["sha256"]}
            for record in records
        ),
        key=lambda record: (record["path"], record["sha256"]),
    )
    canonical = json.dumps(normalized, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def load_training_fingerprint(manifest_path: Path) -> str:
    """Reconstruct the exact full-corpus identity recorded during training."""
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    training_files = manifest.get("training_files")
    if not training_files:
        raise ValueError("Training manifest must contain content-addressed training_files.")
    return canonical_dataset_fingerprint(training_files)


def require_external_benchmark(benchmark: Mapping[str, Any], training_fingerprint: str) -> None:
    """Fail closed when benchmark lineage is missing or overlaps the training corpus."""
    if benchmark.get("dataset_role") != "external_evaluation":
        raise ValueError("Benchmark must declare dataset_role='external_evaluation'.")
    benchmark_fingerprint = benchmark.get("dataset_fingerprint")
    if not benchmark_fingerprint:
        raise ValueError("Benchmark must provide a versioned dataset_fingerprint.")
    if benchmark_fingerprint == training_fingerprint:
        raise ValueError("Benchmark fingerprint matches the full training corpus; promotion is blocked.")


def evaluate_release(
    candidate: Mapping[str, float],
    baseline: Mapping[str, float],
    policy: EvaluationPolicy,
) -> dict[str, bool]:
    """Apply only gates enabled for the declared workload claim."""
    gates: dict[str, bool] = {}
    floor_gates = {
        "instruction": ("instruction_pass_rate", policy.min_instruction_pass_rate),
        "source_support": ("source_support_rate", policy.min_source_support_rate),
        "preference": ("preference_win_rate", policy.min_preference_win_rate),
        "safety": ("safety_pass_rate", policy.min_safety_pass_rate),
    }
    for gate_name, (metric_name, threshold) in floor_gates.items():
        if threshold is not None:
            gates[gate_name] = candidate[metric_name] >= threshold

    ceiling_gates = {
        "latency": ("p95_latency_ms", policy.max_p95_latency_ms),
        "cost": ("cost_per_1k_requests_usd", policy.max_cost_per_1k_requests_usd),
    }
    for gate_name, (metric_name, threshold) in ceiling_gates.items():
        if threshold is not None:
            gates[gate_name] = candidate[metric_name] <= threshold

    if policy.max_perplexity_regression_pct is not None:
        allowed_perplexity = baseline["heldout_perplexity"] * (
            1.0 + policy.max_perplexity_regression_pct / 100.0
        )
        gates["perplexity"] = candidate["heldout_perplexity"] <= allowed_perplexity

    if not gates:
        raise ValueError("At least one release gate must be enabled.")
    return gates


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Return a streaming SHA-256 digest."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_artifact(path: Path) -> str:
    """Digest one checkpoint file or directory deterministically."""
    digest = hashlib.sha256()
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    for file_path in files:
        relative_path = file_path.name if path.is_file() else file_path.relative_to(path).as_posix()
        digest.update(relative_path.encode("utf-8"))
        with file_path.open("rb") as artifact_file:
            for chunk in iter(lambda: artifact_file.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def build_decision_record(
    config: ProductionDecisionConfig,
    benchmark: Mapping[str, Any],
    gates: Mapping[str, bool],
    artifact_digest: str,
    training_fingerprint: str,
) -> dict[str, Any]:
    """Capture the claim, external evidence, decision, and rollback target."""
    promoted = all(gates.values())
    return {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "workload": config.workload,
        "hypothesis": benchmark["hypothesis"],
        "conclusion": "supported" if promoted else "not_supported",
        "decision": "canary" if promoted else "keep_accepted_release",
        "candidate": {
            "name": config.candidate_name,
            "strategy": config.candidate_strategy,
            "artifact": str(config.candidate_artifact),
            "sha256": artifact_digest,
        },
        "rollback": {
            "name": config.rollback_name,
            "artifact": str(config.rollback_artifact),
        },
        "lineage": {
            "dataset_role": benchmark["dataset_role"],
            "evaluation_dataset_fingerprint": benchmark["dataset_fingerprint"],
            "training_dataset_fingerprint": training_fingerprint,
            "code_revision": benchmark["code_revision"],
            "base_model": config.base_model_id,
            "base_revision": config.base_revision,
        },
        "policy": asdict(config.policy),
        "candidate_metrics": benchmark["candidate"],
        "baseline_metrics": benchmark["baseline"],
        "gates": dict(gates),
    }


def write_decision_record(record: Mapping[str, Any], registry_dir: Path) -> Path:
    """Write a content-addressed decision record."""
    registry_dir.mkdir(parents=True, exist_ok=True)
    canonical = json.dumps(record, sort_keys=True, separators=(",", ":"))
    decision_id = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
    output_path = registry_dir / f"decision-{decision_id}.json"
    output_path.write_text(json.dumps(record, indent=2, sort_keys=True), encoding="utf-8")
    return output_path


def required_artifact_groups(strategy: str) -> tuple[tuple[str, ...], ...]:
    """Return alternative required filenames for each parameter strategy."""
    if strategy in {"lora", "qlora"}:
        return (("adapter_config.json",), ("adapter_model.safetensors", "adapter_model.bin"))
    if strategy in {"full", "partial-freeze"}:
        return (("config.json",), ("model.safetensors", "pytorch_model.bin"))
    raise ValueError(f"Unsupported strategy: {strategy}")


def package_approved_release(config: ProductionDecisionConfig, record: Mapping[str, Any]) -> Path:
    """Atomically package only a candidate whose external-evidence decision is canary."""
    if record["decision"] != "canary":
        raise ValueError("Only a candidate with a supported external-evidence decision may be packaged.")
    for alternatives in required_artifact_groups(config.candidate_strategy):
        if not any((config.candidate_artifact / name).is_file() for name in alternatives):
            raise FileNotFoundError(f"Expected one of {alternatives} in {config.candidate_artifact}")

    release_id = f"{config.workload}-{record['candidate']['sha256'][:12]}"
    final_dir = config.release_root / release_id
    staging_dir = config.release_root / f".{release_id}.staging"
    if final_dir.exists() or staging_dir.exists():
        raise FileExistsError(f"Release path already exists for {release_id}")

    config.release_root.mkdir(parents=True, exist_ok=True)
    try:
        artifact_dir = staging_dir / "model"
        shutil.copytree(config.candidate_artifact, artifact_dir)
        release_manifest = {
            "schema_version": 1,
            "decision_record": record,
            "files": {
                path.relative_to(staging_dir).as_posix(): {
                    "bytes": path.stat().st_size,
                    "sha256": sha256_file(path),
                }
                for path in sorted(artifact_dir.rglob("*"))
                if path.is_file()
            },
        }
        (staging_dir / "release-manifest.json").write_text(
            json.dumps(release_manifest, indent=2, sort_keys=True), encoding="utf-8"
        )
        os.replace(staging_dir, final_dir)
        return final_dir
    except Exception:
        shutil.rmtree(staging_dir, ignore_errors=True)
        raise

In [ ]:
def require_disjoint_file_sets(benchmark, training_files):
    """Verify benchmark file lineage and reject any content overlap with training."""
    benchmark_files = benchmark.get("dataset_files")
    if not benchmark_files:
        raise ValueError("Benchmark must include content-addressed dataset_files.")

    computed_fingerprint = canonical_dataset_fingerprint(benchmark_files)
    if benchmark.get("dataset_fingerprint") != computed_fingerprint:
        raise ValueError("Benchmark dataset_fingerprint does not match dataset_files.")

    training_digests = {record["sha256"] for record in training_files}
    benchmark_digests = {record["sha256"] for record in benchmark_files}
    overlap = training_digests & benchmark_digests
    if overlap:
        raise ValueError(
            f"Benchmark overlaps training content in {len(overlap)} file digest(s); promotion is blocked."
        )


production_config = ProductionDecisionConfig()

if RUN_PRODUCTION_DECISION:
    benchmark = json.loads(
        production_config.benchmark_metrics.read_text(encoding="utf-8")
    )
    if not production_config.candidate_artifact.exists():
        raise FileNotFoundError(
            f"Candidate artifact not found: {production_config.candidate_artifact}"
        )
    if not production_config.training_manifest.is_file():
        raise FileNotFoundError(
            f"Training manifest not found: {production_config.training_manifest}"
        )

    training_manifest = json.loads(
        production_config.training_manifest.read_text(encoding="utf-8")
    )
    training_files = training_manifest.get("training_files")
    if not training_files:
        raise ValueError("Training manifest must contain content-addressed training_files.")

    training_fingerprint = canonical_dataset_fingerprint(training_files)
    require_external_benchmark(benchmark, training_fingerprint)
    require_disjoint_file_sets(benchmark, training_files)

    gates = evaluate_release(
        benchmark["candidate"],
        benchmark["baseline"],
        production_config.policy,
    )
    record = build_decision_record(
        production_config,
        benchmark,
        gates,
        sha256_artifact(production_config.candidate_artifact),
        training_fingerprint,
    )
    record_path = write_decision_record(record, production_config.registry_dir)

    print("Hypothesis:", record["hypothesis"])
    print("Training fingerprint:", record["lineage"]["training_dataset_fingerprint"])
    print("External benchmark fingerprint:", record["lineage"]["evaluation_dataset_fingerprint"])
    for gate_name, passed in gates.items():
        print(f"{gate_name:>15}: {'PASS' if passed else 'FAIL'}")
    print("Conclusion:", record["conclusion"])
    print("Decision:", record["decision"])
    print("Decision record:", record_path)

    if record["decision"] == "canary":
        release_dir = package_approved_release(production_config, record)
        print("Immutable canary release:", release_dir)
else:
    print(
        "Production decision workflow is disabled. Supply an external versioned benchmark with "
        "content-addressed files, product-owned thresholds, candidate and rollback artifacts, "
        "and the Part 1 training manifest before enabling it. Training overlap is rejected."
    )

---

## End of Part 3: Evidence Before Preference

The durable evaluation habit is not a formula. It is an order of operations:

1. Name one candidate's job and falsifiable claim.
2. Freeze independent cases and their lineage.
3. Preserve case records and failure reasons.
4. Aggregate the metric that matches the claim.
5. Apply requirements chosen before seeing candidate results.
6. Compare cost only among candidates that preserve the required behavior.

The full-corpus artifacts from Parts 1 and 2 remain **mechanism artifacts**. Part 3 can open their scorers and decision logic, but no same-Aria calculation can promote them.

Continue to **[Part 4: GPU Practice](04-llm-finetuning-practice.ipynb)** for a different experiment: Part 4 trains new LoRA continued-pretraining candidates from chapter-disjoint training splits, selects checkpoints on validation chapters, and opens untouched test chapters only after selection. It provides controlled CPT practice across eight novels; it does not retroactively validate the full-corpus CPT, SFT, DPO, or parameter-strategy teaching artifacts.

That boundary is the final lesson: when an earlier experiment cannot answer the production question, design the next experiment rather than asking a contaminated result to say more.